In [9]:
import torch
import torch.nn as nn

In [10]:
import sys
from pathlib import Path

In [11]:
PROJECT_ROOT = Path("/home/hmx42/Projects/Kronos_projest")
MY_CODE_DIR = PROJECT_ROOT / "my_code"
from data import (
    default_csv_path,
    prepare_prediction_window,
    save_processed_window,
)

In [12]:
window = prepare_prediction_window(
    csv_path=default_csv_path(),
    lookback=400,
    pred_len=120,
    clip=5.0,
)

x_norm = window.x_norm

x_norm.shape

(400, 6)

In [13]:
import torch

x_tensor = torch.from_numpy(x_norm).float().unsqueeze(0)
x_tensor.shape

torch.Size([1, 400, 6])

# 1. RMSNorm
# 2. FeedForward
# 3. MultiHeadAttentionWithRoPE
# 4. TransformerBlock
# 5. BinarySphericalQuantizer
# 6. BSQuantizer
# 7. Tokenizer

In [15]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # x: [B, T, D]
        norm_x = x * torch.rsqrt(torch.mean(x * x, dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight

In [16]:
import torch.nn.functional as F

class FeedForward(nn.Module):
    def __init__(self, d_model, ff_dim, ffn_dropout_p=0.0):
        super().__init__()

        self.w1 = nn.Linear(d_model, ff_dim, bias=False)
        self.w3 = nn.Linear(d_model, ff_dim, bias=False)
        self.w2 = nn.Linear(ff_dim, d_model, bias=False)
        self.ffn_dropout = nn.Dropout(ffn_dropout_p)

    def forward(self, x):
        # x: [B, T, d_model]

        gate = F.silu(self.w1(x))
        value = self.w3(x)

        hidden = gate * value
        out = self.w2(hidden)
        out = self.ffn_dropout(out)

        return out